# Hoja de ejercicios 1: Bioinformática con Biopython

**Autores:** Laura Aguiar Pérez y Amai Suarez Navarro

Este notebook reúne los cinco ejercicios ya ejecutados, con su salida. El código es el mismo que el de los scripts `ejercicio*.py` de la carpeta, que también se pueden ejecutar por separado. Los archivos generados se guardan en `resultados/`.

**Requisitos:** `pip install -r requirements.txt` y conexión a internet (ejercicios 2 a 5).

## Configuración común

El NCBI pide identificarse con un correo en `Entrez.email`. `comun.py` lo lee de la variable de entorno `NCBI_EMAIL` y define las carpetas `data/` y `resultados/`. Cada ejercicio lo importa con `from comun import ...`.

## Ejercicio 1: Porcentaje GC

1. Se lee el FASTA con `SeqIO.parse`, que admite varias secuencias.
2. El %GC se calcula con `gc_fraction` (equivale a (G + C) / longitud × 100).
3. Cada secuencia se traduce a aminoácidos. Si su longitud no es múltiplo de 3 se recorta al último codón completo (así Biopython no da el aviso de *partial codon*), y con `to_stop=True` la traducción termina en el primer codón de parada.
4. Las proteínas se escriben en un FASTA nuevo con el mismo `id` y `description` que las secuencias originales.

`data/secuencias_adn.fasta` tiene 4 secuencias de ejemplo; la última es el inicio real del gen *HBB* humano, y su traducción coincide con el inicio de la β-globina (`MVHLTPEEKS...`).

In [1]:
import os

from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import gc_fraction

from comun import DATOS, RESULTADOS


def main(entrada=DATOS / "secuencias_adn.fasta",
         salida=RESULTADOS / "ejercicio1_proteinas.fasta"):
    proteinas = []
    for registro in SeqIO.parse(entrada, "fasta"):
        gc = gc_fraction(registro.seq) * 100
        print(f"{registro.id}: {len(registro.seq)} nt, GC = {gc:.2f}%")

        # Se recorta al último codón completo para evitar avisos de Biopython
        # y se traduce hasta el primer codón de parada (el péptido real)
        longitud = len(registro.seq) - len(registro.seq) % 3
        proteina = registro.seq[:longitud].translate(to_stop=True)
        print(f"  Proteína: {proteina}")

        proteinas.append(
            SeqRecord(proteina, id=registro.id, description=registro.description)
        )

    SeqIO.write(proteinas, salida, "fasta")
    print(f"\n{len(proteinas)} secuencias de aminoácidos guardadas en {os.path.relpath(salida)}")


main()

seq1: 69 nt, GC = 52.17%
  Proteína: MAIVMGR
seq2: 57 nt, GC = 89.47%
  Proteína: MAAAGAGGPAEPRGGCGP
seq3: 54 nt, GC = 3.70%
  Proteína: MKFKIFNKEIIKFIKNY
seq4: 93 nt, GC = 59.14%
  Proteína: MVHLTPEEKSAVTALWGKVNVDEVGGEALGR

4 secuencias de aminoácidos guardadas en resultados\ejercicio1_proteinas.fasta


In [2]:
# Comprobación del FASTA generado
print((RESULTADOS / "ejercicio1_proteinas.fasta").read_text())

>seq1 Fragmento codificante de prueba 1
MAIVMGR
>seq2 Fragmento codificante de prueba 2 rico en GC
MAAAGAGGPAEPRGGCGP
>seq3 Fragmento codificante de prueba 3 rico en AT
MKFKIFNKEIIKFIKNY
>seq4 Inicio del gen HBB humano (beta-globina)
MVHLTPEEKSAVTALWGKVNVDEVGGEALGR



## Ejercicio 2: El sarampión

- `esearch` en la base de datos `nucleotide`. Para usar la base de datos **secundaria** (RefSeq, secuencias curadas) se añade `AND refseq[filter]` al término de búsqueda.
- Se toma `IdList[0]`, se descarga con `efetch` en formato FASTA y se muestran la descripción y los primeros 100 nucleótidos.

El resultado es el genoma de referencia del virus, **NC_001498.1** (unos 15 900 nt, ARN monocatenario de polaridad negativa).

In [3]:
from Bio import Entrez, SeqIO

from comun import RESULTADOS

# 1. Búsqueda en "nucleotide" restringida a la base de datos secundaria RefSeq
busqueda = Entrez.esearch(
    db="nucleotide",
    term="Measles morbillivirus[Organism] AND refseq[filter]",
)
resultado = Entrez.read(busqueda)
busqueda.close()

# 2. Primer ID de la lista
id_genoma = resultado["IdList"][0]
print(f"Resultados encontrados: {resultado['Count']}, usando el ID {id_genoma}")

# 3. Descarga del registro en formato FASTA
descarga = Entrez.efetch(db="nucleotide", id=id_genoma, rettype="fasta", retmode="text")
registro = SeqIO.read(descarga, "fasta")
descarga.close()

SeqIO.write(registro, RESULTADOS / "sarampion_genoma.fasta", "fasta")

# 4. Descripción y primeros 100 nucleótidos
print(f"Descripción: {registro.description}")
print(f"Longitud: {len(registro.seq)} nt")
print(f"Primeros 100 nucleótidos:\n{registro.seq[:100]}")

Resultados encontrados: 1, usando el ID 9626945
Descripción: NC_001498.1 Measles virus, complete genome
Longitud: 15894 nt
Primeros 100 nucleótidos:
ACCAAACAAAGTTGGGTAAGGATAGATCAATCAATGATCATATTCTAGTACACTTAGGATTCAAGATCCTATTATCAGGGACAAGAGCAGGATTAGGGAT


## Ejercicio 3: La COVID-19

Se descarga NC_045512 en GenBank y se recorren sus `features`, quedándonos solo con las de tipo `gene`. El nombre sale de `qualifiers.get("gene")`.

Biopython guarda las coordenadas empezando en 0 y con el final exclusivo (como Python). Para mostrarlas como en el NCBI se suma 1 al inicio.

In [4]:
from Bio import Entrez, SeqIO

from comun import RESULTADOS

ACCESO = "NC_045512"

# 1. Descarga en formato GenBank
descarga = Entrez.efetch(db="nucleotide", id=ACCESO, rettype="gb", retmode="text")
registro = SeqIO.read(descarga, "genbank")
descarga.close()
SeqIO.write(registro, RESULTADOS / f"{ACCESO}.gb", "genbank")

print(f"{registro.id}: {registro.description}\n")

# 2-3. Recorrer las características quedándonos solo con los genes
genes = [f for f in registro.features if f.type == "gene"]

# 4-5. Nombre del gen y coordenadas (inicio en base 1, como en GenBank)
print(f"{'Nº':>3}  {'Gen':<10}{'Inicio':>8}{'Fin':>8}")
for i, gen in enumerate(genes, start=1):
    nombre = gen.qualifiers.get("gene", ["(sin nombre)"])[0]
    inicio = int(gen.location.start) + 1
    fin = int(gen.location.end)
    print(f"{i:>3}  {nombre:<10}{inicio:>8}{fin:>8}")

print(f"\nTotal de genes: {len(genes)}")

NC_045512.2: Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1, complete genome

 Nº  Gen         Inicio     Fin
  1  ORF1ab         266   21555
  2  S            21563   25384
  3  ORF3a        25393   26220
  4  E            26245   26472
  5  M            26523   27191
  6  ORF6         27202   27387
  7  ORF7a        27394   27759
  8  ORF7b        27756   27887
  9  ORF8         27894   28259
 10  N            28274   29533
 11  ORF10        29558   29674

Total de genes: 11


El genoma tiene **11 genes**: *ORF1ab* (poliproteína con la replicasa), *S* (espícula, la proteína que reconoce el receptor ACE2), *ORF3a*, *E* (envoltura), *M* (membrana), *ORF6*, *ORF7a*, *ORF7b*, *ORF8*, *N* (nucleocápside) y *ORF10*.

## Ejercicio 4: Bacterias, a la caza de proteínas de resistencia

**Nota sobre el código de acceso:** el enunciado indica `NC_001371`, pero en el NCBI ese código corresponde al plásmido ColE1 de *Escherichia coli* (y es un registro CON, sin secuencia). El plásmido pT181 de *Staphylococcus aureus* es **`NC_001393`**, que es el que se usa aquí.

- Se descarga con `rettype="gbwithparts"` para que el registro traiga la secuencia completa.
- Se filtran las features `CDS` y se extrae su ADN con `.extract()`, que respeta la hebra y las uniones (un CDS cruza el origen del plásmido circular).
- Se traduce con la **tabla 11** (código genético bacteriano). Con `cds=True` los codones de inicio alternativos (TTG, GTG) se traducen como M.
- Como control, la traducción se compara con la anotada en el GenBank (`/translation`).
- Las proteínas cuya descripción contiene términos de resistencia se marcan automáticamente.

En el RefSeq actual *tet(K)* está anotado como **pseudogén** (`/pseudo`): tiene un codón de parada interno, así que no trae traducción anotada. Se traduce hasta esa primera parada.

In [5]:
from Bio import Entrez, SeqIO
from Bio.Data import CodonTable
from Bio.SeqRecord import SeqRecord

from comun import RESULTADOS

ACCESO = "NC_001393"

# Términos habituales en las anotaciones de genes de resistencia
PALABRAS_RESISTENCIA = (
    "resistance", "efflux", "tetracycline", "lactamase",
    "aminoglycoside", "chloramphenicol", "erythromycin", "vancomycin",
)

# "gbwithparts" garantiza que el registro incluya la secuencia completa
descarga = Entrez.efetch(db="nucleotide", id=ACCESO, rettype="gbwithparts", retmode="text")
plasmido = SeqIO.read(descarga, "genbank")
descarga.close()
SeqIO.write(plasmido, RESULTADOS / f"{ACCESO}.gb", "genbank")

print(f"{plasmido.id}: {plasmido.description} ({len(plasmido.seq)} pb)\n")

proteinas = []
for cds in (f for f in plasmido.features if f.type == "CDS"):
    gen = cds.qualifiers.get("gene", ["?"])[0]
    producto = cds.qualifiers.get("product", ["desconocido"])[0]
    id_proteina = cds.qualifiers.get("protein_id", [gen])[0]

    # extract() respeta la hebra y las uniones; tabla 11 = código bacteriano.
    # cds=True traduce los codones de inicio alternativos (TTG, GTG) como M.
    adn = cds.extract(plasmido.seq)
    try:
        proteina = adn.translate(table=11, cds=True)
    except CodonTable.TranslationError:
        # Pseudogenes: tienen paradas internas, se traduce hasta la primera.
        # El codón de inicio (aquí TTG) se pone como M, igual que con cds=True
        proteina = adn.translate(table=11, to_stop=True)
        proteina = "M" + proteina[1:]

    # Comprobación con la traducción que trae anotada el propio GenBank
    anotada = cds.qualifiers.get("translation", [None])[0]
    if "pseudo" in cds.qualifiers:
        coincide = "pseudogén, sin traducción anotada"
    else:
        coincide = "sí" if anotada == str(proteina) else "no"

    es_resistencia = any(p in producto.lower() for p in PALABRAS_RESISTENCIA)
    resistencia = "  <-- PROTEÍNA DE RESISTENCIA" if es_resistencia else ""
    print(f"{id_proteina} | gen {gen} | {producto}{resistencia}")
    print(f"  Posición {cds.location} | {len(proteina)} aa | coincide con la anotación: {coincide}")
    print(f"  {proteina}\n")

    proteinas.append(
        SeqRecord(proteina, id=id_proteina, description=f"{gen} {producto} [{plasmido.id}]")
    )

salida = RESULTADOS / "pT181_proteinas.fasta"
SeqIO.write(proteinas, salida, "fasta")
print(f"{len(proteinas)} proteínas guardadas en resultados/{salida.name}")

NC_001393.1: Staphylococcus aureus plasmid pT181, complete sequence (4439 pb)

WP_010889937.1 | gen ? | replication initiation factor domain-containing protein
  Posición join{[4288:4439](+), [0:794](+)} | 314 aa | coincide con la anotación: sí
  MYKNNHANHSNHLENHDLDNFSKTGYSNSRLDAHTVCISDPKLSFDAMTIVGNLNRDNAQALSKFMSVEPQIRLWDILQTKFKAKALQEKVYIEYDKVKADSWDRRNMRIEFNPNKLTRDEMIWLKQNIISYMEDDGFTRLDLAFDFEEDLSDYYAMSDKAVKKTIFYGRNGKPETKYFGVRDSNRFIRIYNKKQERKDNADAEVMSEHLWRVEIELKRDMVDYWNDCFSDLHILQPDWKTIQRTADRAIVFMLLSDEEEWGKLHRNSRTKYKNLIKEISPVDLTDLMKSTLKANEKQLQKQIDFWQHEFKFWK

tet(K) | gen tet(K) | tetracycline efflux MFS transporter Tet(K)  <-- PROTEÍNA DE RESISTENCIA
  Posición [955:2335](+) | 295 aa | coincide con la anotación: pseudogén, sin traducción anotada
  MFSLYKKFKGLFYSVLFWLCILSFFSVLNEMVLNVSLPDIANHFNTTPGITNWVNTAYMLTFSIGTAVYGKLSDYINIKKLLIIGISLSCLGSLIAFIGHNHFFILIFGRLVQGVGSAAFPSLIMVVVARNITRKKQGKAFGFIGSIVALGEGLGPSIGGIIAHYIHWSYLLILPMITIVTIPFLIKVMVPGKSTKNTLDIVGIVLMSISIICFMLFTTNYNWTFLILFTIFFVIFIKHISRVS

**Conclusión:** el plásmido codifica **Tet(K)**, una bomba de expulsión de la familia MFS que saca la tetraciclina de la célula. Por tanto, esta bacteria se defiende de la **tetraciclina**. Las otras dos CDS no son de resistencia: la proteína de iniciación de la replicación (RepC, que permite copiar el plásmido) y la relaxasa MobV (que lo moviliza para transferirlo a otras bacterias).

## Ejercicio 5: Algunas proteínas "célebres"

Proteínas elegidas: **hemoglobina**, **mioglobina** y **oxitocina** (humanas). Las dos primeras son globinas que transportan o almacenan oxígeno; la oxitocina sirve de contraste porque es una hormona peptídica.

El script descarga de UniProt la ficha (JSON) y el FASTA de cada una, y calcula su masa molecular y su punto isoeléctrico con `ProteinAnalysis`. El informe completo con la información de cada proteína y las fuentes está en `docs/ejercicio5_proteinas.md`; abajo va un resumen.

In [6]:
import json
from urllib.request import urlopen

from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

from comun import RESULTADOS

UNIPROT = "https://rest.uniprot.org/uniprotkb"

PROTEINAS = {
    "Hemoglobina (subunidad beta)": "P68871",
    "Mioglobina": "P02144",
    "Oxitocina (precursor oxitocina-neurofisina 1)": "P01178",
}


def descargar_fasta(acceso):
    destino = RESULTADOS / f"{acceso}.fasta"
    with urlopen(f"{UNIPROT}/{acceso}.fasta") as respuesta:
        destino.write_bytes(respuesta.read())
    return SeqIO.read(destino, "fasta")


def descargar_ficha(acceso):
    with urlopen(f"{UNIPROT}/{acceso}.json") as respuesta:
        return json.load(respuesta)


def funcion(ficha):
    for comentario in ficha.get("comments", []):
        if comentario["commentType"] == "FUNCTION":
            return comentario["texts"][0]["value"]
    return "(sin descripción)"


registros = []
for nombre, acceso in PROTEINAS.items():
    ficha = descargar_ficha(acceso)
    registro = descargar_fasta(acceso)
    registros.append(registro)

    analisis = ProteinAnalysis(str(registro.seq))
    gen = ficha["genes"][0]["geneName"]["value"]
    nombre_uniprot = ficha["proteinDescription"]["recommendedName"]["fullName"]["value"]

    print("=" * 70)
    print(f"{nombre}  [UniProt {acceso}]")
    print(f"Nombre oficial: {nombre_uniprot} | Gen: {gen} | "
          f"Organismo: {ficha['organism']['scientificName']}")
    print(f"Longitud: {len(registro.seq)} aa | "
          f"Masa molecular: {analisis.molecular_weight() / 1000:.1f} kDa | "
          f"Punto isoeléctrico: {analisis.isoelectric_point():.2f}")
    print(f"Función: {funcion(ficha)}")
    print(f"Secuencia:\n{registro.seq}\n")

SeqIO.write(registros, RESULTADOS / "ejercicio5_proteinas.fasta", "fasta")
print("FASTA combinado guardado en resultados/ejercicio5_proteinas.fasta")

Hemoglobina (subunidad beta)  [UniProt P68871]
Nombre oficial: Hemoglobin subunit beta | Gen: HBB | Organismo: Homo sapiens
Longitud: 147 aa | Masa molecular: 16.0 kDa | Punto isoeléctrico: 6.74
Función: Involved in oxygen transport from the lung to the various peripheral tissues
Secuencia:
MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH

Mioglobina  [UniProt P02144]
Nombre oficial: Myoglobin | Gen: MB | Organismo: Homo sapiens
Longitud: 154 aa | Masa molecular: 17.2 kDa | Punto isoeléctrico: 7.14
Función: Monomeric heme protein which primary function is to store oxygen and facilitate its diffusion within muscle tissues. Reversibly binds oxygen through a pentacoordinated heme iron and enables its timely and efficient release as needed during periods of heightened demand (PubMed:30918256, PubMed:34679218). Depending on the oxidative conditions of tissues and cells, and in addition to its 

| Proteína | UniProt | Gen | Función principal |
|---|---|---|---|
| Hemoglobina, subunidad β | P68871 | *HBB* (cr. 11) | Transporta O₂ en los glóbulos rojos. Tetrámero α₂β₂ con 4 grupos hemo y unión cooperativa (curva sigmoidea). La mutación Glu6Val causa la anemia falciforme. |
| Mioglobina | P02144 | *MB* (cr. 22) | Almacena O₂ en el músculo. Monómero con un solo hemo (curva hiperbólica, más afinidad que la hemoglobina). Primera estructura 3D resuelta por rayos X (Kendrew, 1958). |
| Oxitocina-neurofisina 1 | P01178 | *OXT* (cr. 20) | Precursor de 125 aa del que se corta el nonapéptido **CYIQNCPLG**. Contracciones del parto, eyección de leche y vínculo social. |